In [5]:
import os
import glob
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from konlpy.tag import Mecab
from tqdm import tqdm
import time

# ---------------------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------------------
DATA_DIR = "data/대법원_판례" # JSON 파일들이 있는 폴더
print(f">>> '{DATA_DIR}' 폴더에서 데이터 로딩 중...")

def load_custom_dataset(data_dir):
    json_files = glob.glob(os.path.join(data_dir, "**", "*.json"), recursive=True)
    texts = []
    labels = []
    
    print(f"   발견된 JSON 파일 수: {len(json_files)}개")
    
    for file_path in tqdm(json_files):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # [주의] JSON 구조에 따라 키 이름이 다를 수 있음. 
            # 일단 사용자님이 말씀하신 "판례내용", "사건명" 키를 기준으로 작성함.
            # 만약 데이터가 {'판례정보': {'판례내용': ...}} 처럼 중첩되어 있다면 수정 필요.
            
            # 1. 텍스트 (판례내용) 추출
            # (데이터에 따라 '판례내용'이 없을 수도 있으므로 get 사용)
            content = data.get('판례내용', "")
            if not content: continue # 내용 없으면 스킵
            
            # 2. 라벨 (사건명) 추출
            label = data.get('사건명', "")
            if not label: continue
            
            # HTML 태그 등 간단한 정제 (필요시)
            content = content.replace("<br/>", " ").replace("\n", " ")
            
            texts.append(content)
            labels.append(label)
            
        except Exception as e:
            # 파일 깨짐 등 오류 발생 시 무시하고 진행
            continue
            
    return pd.DataFrame({'text': texts, 'label': labels})

# 데이터 로드 실행
df_raw = load_custom_dataset(DATA_DIR)
print(f"   로딩된 전체 데이터 수: {len(df_raw)}건")

if len(df_raw) == 0:
    print("[Error] 로딩된 데이터가 없습니다. JSON 파일의 키 값('사건명', '판례내용')을 확인해주세요.")
    exit()

# ---------------------------------------------------------
# 2. 데이터 필터링 (Top N 클래스만 사용)
# ---------------------------------------------------------
# 사건명이 너무 다양하면(수백 개) 분류가 안 되므로, 데이터가 많은 상위 10개만 추림
TOP_N_CLASSES = 10
top_classes = df_raw['label'].value_counts().head(TOP_N_CLASSES).index.tolist()

print(f"\n>>> 실험을 위해 데이터가 가장 많은 상위 {TOP_N_CLASSES}개 사건명만 사용합니다.")
print(f"   대상 사건명: {top_classes}")

# 필터링
df_filtered = df_raw[df_raw['label'].isin(top_classes)].copy()
print(f"   필터링 후 데이터 수: {len(df_filtered)}건")

# 학습/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['text'], df_filtered['label'], test_size=0.2, random_state=42, stratify=df_filtered['label']
)

# ---------------------------------------------------------
# 3. 실험 환경 설정 (형태소 분석기 & 불용어 로드)
# ---------------------------------------------------------
mecab = Mecab()

def tokenizer(text):
    # 명사만 추출 (가장 깔끔한 비교를 위해)
    return [word for word, pos in mecab.pos(text) if pos.startswith('N')]

def load_stopwords_set(filepath):
    try:
        df = pd.read_csv(filepath)
        return set(df['word'].astype(str).tolist())
    except:
        return set()

# 비교할 실험군 정의
stopword_sets = {
    "1. Baseline (불용어 없음)": set(),
    "2. Exp A (빈도_Zipf)": load_stopwords_set("stopwords_exp_A_zipf.csv"),
    "3. Exp B (엔트로피)": load_stopwords_set("stopwords_exp_B_entropy.csv"),
    "4. Exp C (안정성_MPVP)": load_stopwords_set("stopwords_exp_C_mpvp.csv"),
    "5. Exp D (통합_Union)": load_stopwords_set("stopwords_exp_D_union.csv")
}

# ---------------------------------------------------------
# 4. 비교 평가 루프
# ---------------------------------------------------------
results = []
print("\n>>> 불용어 리스트별 성능 비교 시작 (Logistic Regression)")
print("-" * 70)
print(f"{'Experiment':<25} | {'Acc':<8} | {'F1-Score':<8} | {'Time':<8} | {'Stopwords'}")
print("-" * 70)

for name, stop_set in stopword_sets.items():
    # 불용어 리스트 변환
    stop_list = list(stop_set) if stop_set else None
    
    # 파이프라인: TF-IDF -> 로지스틱 회귀
    # max_features=5000: 속도를 위해 단어 수 제한 (필요시 조절)
    pipeline = Pipeline([
        ('vect', TfidfVectorizer(tokenizer=tokenizer, stop_words=stop_list, max_features=5000)),
        ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
    ])
    
    start_time = time.time()
    
    # 학습
    pipeline.fit(X_train, y_train)
    
    # 예측
    pred = pipeline.predict(X_test)
    
    # 평가
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='weighted') # 클래스 불균형 고려
    elapsed = time.time() - start_time
    
    print(f"{name:<25} | {acc:.4f}   | {f1:.4f}   | {elapsed:.1f}s   | {len(stop_set)} words")
    
    results.append({
        'Experiment': name,
        'Stopwords_Count': len(stop_set),
        'Accuracy': acc,
        'F1_Score': f1,
        'Time_Sec': elapsed
    })

# ---------------------------------------------------------
# 5. 결과 저장 및 출력
# ---------------------------------------------------------
result_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
result_df.to_csv("final_experiment_results.csv", index=False, encoding='utf-8-sig')

print("-" * 70)
print("\n[최종 순위]")
print(result_df)
print("\n파일 저장 완료: final_experiment_results.csv")

>>> 'data/대법원_판례' 폴더에서 데이터 로딩 중...
   발견된 JSON 파일 수: 5472개


100%|██████████| 5472/5472 [00:01<00:00, 4293.13it/s]
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


   로딩된 전체 데이터 수: 5472건

>>> 실험을 위해 데이터가 가장 많은 상위 10개 사건명만 사용합니다.
   대상 사건명: ['손해배상(기)', '부당이득금', '구상금', '임금', '소유권이전등기', '사해행위취소', '배당이의', '보험금', '대여금', '등록무효(특)']
   필터링 후 데이터 수: 842건

>>> 불용어 리스트별 성능 비교 시작 (Logistic Regression)
----------------------------------------------------------------------
Experiment                | Acc      | F1-Score | Time     | Stopwords
----------------------------------------------------------------------
1. Baseline (불용어 없음)      | 0.7101   | 0.7048   | 5.1s   | 0 words


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


2. Exp A (빈도_Zipf)        | 0.6686   | 0.6623   | 3.4s   | 167 words


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['고등', '상고이', '심', '유서'] not in stop_words.
  warnings.warn(


3. Exp B (엔트로피)           | 0.6805   | 0.6759   | 3.3s   | 226 words


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


4. Exp C (안정성_MPVP)       | 0.7101   | 0.7046   | 2.5s   | 54 words


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['고등', '상고이', '심', '유서'] not in stop_words.
  warnings.warn(


5. Exp D (통합_Union)       | 0.6746   | 0.6702   | 3.5s   | 286 words
----------------------------------------------------------------------

[최종 순위]
             Experiment  Stopwords_Count  Accuracy  F1_Score  Time_Sec
0  1. Baseline (불용어 없음)                0  0.710059  0.704795  5.054035
3   4. Exp C (안정성_MPVP)               54  0.710059  0.704612  2.527267
2       3. Exp B (엔트로피)              226  0.680473  0.675894  3.307680
4   5. Exp D (통합_Union)              286  0.674556  0.670188  3.500292
1    2. Exp A (빈도_Zipf)              167  0.668639  0.662298  3.390842

파일 저장 완료: final_experiment_results.csv


In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from konlpy.tag import Mecab
import time

# ---------------------------------------------------------
# 1. 데이터 로드 (Top 10 사건 필터링된 데이터 사용)
# ---------------------------------------------------------
# (앞서 로드한 df_filtered 데이터프레임을 그대로 사용한다고 가정)
# 만약 커널이 재시작되었다면 이전 코드의 '데이터 로드' 부분을 먼저 실행해주세요!
if 'df_filtered' not in locals():
    print("[주의] 데이터가 메모리에 없습니다. 이전 단계의 '데이터 로드' 코드를 먼저 실행해주세요.")
    # (여기서는 실행되지 않도록 exit 처리하거나, 사용자가 인지하도록 메시지만 띄움)

# 학습/테스트 분리 (동일한 시드 사용)
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered['text'], df_filtered['label'], test_size=0.2, random_state=42, stratify=df_filtered['label']
)

# ---------------------------------------------------------
# 2. 불용어 리스트 준비 (교집합 추가)
# ---------------------------------------------------------
def load_stopwords_set(filepath):
    try:
        df = pd.read_csv(filepath)
        return set(df['word'].astype(str).tolist())
    except:
        return set()

# 기존 리스트 로드
set_a = load_stopwords_set("stopwords_exp_A_zipf.csv")
set_b = load_stopwords_set("stopwords_exp_B_entropy.csv")
set_c = load_stopwords_set("stopwords_exp_C_mpvp.csv")
set_d = load_stopwords_set("stopwords_exp_D_union.csv")

# [신규] Exp E: 교집합 (Intersection)
# A(빈도) AND B(엔트로피) AND C(안정성) 조건을 모두 만족하는 단어
set_e = set_a.intersection(set_b).intersection(set_c)

# 교집합 파일 저장 (분석용)
pd.DataFrame({'word': list(set_e)}).to_csv("stopwords_exp_E_intersection.csv", index=False)
print(f"\n>>> [신규] 교집합(Exp E) 리스트 생성 완료: 총 {len(set_e)}개 단어")
print(f"    (예시: {list(set_e)[:10]})")

# 실험군 정의
stopword_sets = {
    "1. Baseline (None)": set(),
    "2. Exp A (Zipf)": set_a,
    "3. Exp B (Entropy)": set_b,
    "4. Exp C (MP-VP)": set_c,
    "5. Exp D (Union)": set_d,
    "6. Exp E (Intersect)": set_e  # <-- 추가됨
}

# ---------------------------------------------------------
# 3. 벤치마크 실행
# ---------------------------------------------------------
mecab = Mecab()
def tokenizer(text):
    return [word for word, pos in mecab.pos(text) if pos.startswith('N')]

results = []
print("\n>>> 불용어 리스트별 성능 비교 시작 (교집합 포함)")
print("-" * 75)
print(f"{'Experiment':<25} | {'Acc':<8} | {'F1':<8} | {'Time':<8} | {'Count'}")
print("-" * 75)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    
    # 파이프라인 (TF-IDF -> Logistic Regression)
    pipeline = Pipeline([
        ('vect', TfidfVectorizer(tokenizer=tokenizer, stop_words=stop_list, max_features=5000)),
        ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
    ])
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='weighted')
    elapsed = time.time() - start_time
    
    print(f"{name:<25} | {acc:.4f}   | {f1:.4f}   | {elapsed:.1f}s   | {len(stop_set)}")
    
    results.append({
        'Experiment': name,
        'Count': len(stop_set),
        'Accuracy': acc,
        'F1_Score': f1,
        'Time': elapsed
    })

# ---------------------------------------------------------
# 4. 결과 요약 및 정성 분석 (리스트 뜯어보기)
# ---------------------------------------------------------
result_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print("\n[최종 순위]")
print(result_df)

# 교집합(Exp E)에는 도대체 뭐가 들어있는지 확인
print("\n" + "="*50)
print(f"[정성 분석] 만장일치 불용어 (Exp E, {len(set_e)}개) 목록")
print("="*50)
print(sorted(list(set_e)))


>>> [신규] 교집합(Exp E) 리스트 생성 완료: 총 44개 단어
    (예시: ['판결', '경우', '취지', '법리', '내용', '제출', '사실', '비추', '정하', '영향'])

>>> 불용어 리스트별 성능 비교 시작 (교집합 포함)
---------------------------------------------------------------------------
Experiment                | Acc      | F1       | Time     | Count
---------------------------------------------------------------------------


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1. Baseline (None)        | 0.7101   | 0.7048   | 3.6s   | 0


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


2. Exp A (Zipf)           | 0.6686   | 0.6623   | 2.5s   | 167


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['고등', '상고이', '심', '유서'] not in stop_words.
  warnings.warn(


... (3 outputs left out)

/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['고등', '상고이', '심', '유서'] not in stop_words.
  warnings.warn(


5. Exp D (Union)          | 0.6746   | 0.6702   | 2.5s   | 286


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


6. Exp E (Intersect)      | 0.7101   | 0.7046   | 3.4s   | 44

[최종 순위]
             Experiment  Count  Accuracy  F1_Score      Time
0    1. Baseline (None)      0  0.710059  0.704795  3.590583
3      4. Exp C (MP-VP)     54  0.710059  0.704612  2.533092
5  6. Exp E (Intersect)     44  0.710059  0.704612  3.408044
2    3. Exp B (Entropy)    226  0.680473  0.675894  2.500815
4      5. Exp D (Union)    286  0.674556  0.670188  2.467042
1       2. Exp A (Zipf)    167  0.668639  0.662298  2.476293

[정성 분석] 만장일치 불용어 (Exp E, 44개) 목록
['경우', '관계', '관련', '관하', '관한', '규정', '내용', '다음', '대법관', '대법원', '대하', '대한', '따라', '목적', '범위', '법률', '법리', '비추', '사실', '사정', '선고', '아니', '여부', '영향', '원심', '원심판결', '위반', '위하', '위한', '의견', '이유', '이하', '인정', '정하', '제출', '주장', '참조', '취지', '판결', '판단', '포함', '필요', '한다', '해당']


### 일반 불용어 목록 추가 테스트

In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from konlpy.tag import Mecab
import time
import os

# ---------------------------------------------------------
# 0. 준비: 기존 변수 확인 (이전 셀이 실행되어 있어야 함)
# ---------------------------------------------------------
if 'X_train' not in locals() or 'set_e' not in locals():
    print("[주의] 이전 단계의 데이터 로드 및 집합 생성 코드를 먼저 실행해주세요!")
    # (실제 실행 시에는 윗 블록 코드들이 메모리에 있다고 가정)

# ---------------------------------------------------------
# 1. 일반 불용어 목록(stopwords.txt) 로드
# ---------------------------------------------------------
GENERAL_STOPWORD_FILE = "stopwords.txt"

# (테스트용) 만약 파일이 없으면 예시로 하나 만듭니다.
if not os.path.exists(GENERAL_STOPWORD_FILE):
    print(f"'{GENERAL_STOPWORD_FILE}'이 없어서 테스트용 파일을 생성합니다.")
    dummy_stopwords = ["이", "그", "저", "것", "수", "등", "들", "및", "에", "의", "를", "가", "은", "는"]
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        f.write("\n".join(dummy_stopwords))

def load_general_stopwords(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            # 공백 제거하고 빈 줄 제외
            return set([line.strip() for line in f if line.strip()])
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return set()

set_general = load_general_stopwords(GENERAL_STOPWORD_FILE)
print(f">>> 일반 불용어(stopwords.txt) 로드 완료: {len(set_general)}개")

# ---------------------------------------------------------
# 2. 실험군 확장 (Hybrid 전략 추가)
# ---------------------------------------------------------
# 기존 세트 (메모리에 있는 변수 활용)
# set_a (빈도), set_b (엔트로피), set_c (MP-VP), set_d (Union), set_e (Intersect)

stopword_sets = {
    # 1. 기존 단독 실험
    "1. Baseline (None)": set(),
    "2. Exp A (Zipf)": set_a,
    "3. Exp B (Entropy)": set_b,
    "4. Exp C (MP-VP)": set_c,
    "5. Exp E (Intersect)": set_e, # D(Union)는 성능 안 좋아서 제외하거나 포함 선택
    
    # 2. 일반 불용어 단독 실험
    "6. General Only": set_general,
    
    # 3. 교차(Hybrid) 실험: 일반 + 법률특화
    "7. General + Exp A": set_general.union(set_a),
    "8. General + Exp B": set_general.union(set_b),
    "9. General + Exp C": set_general.union(set_c),
    "10. General + Exp E": set_general.union(set_e)
}

# ---------------------------------------------------------
# 3. 대규모 벤치마크 실행
# ---------------------------------------------------------
mecab = Mecab()
def tokenizer(text):
    return [word for word, pos in mecab.pos(text) if pos.startswith('N')]

results = []
print("\n>>> [확장 실험] 일반 vs 법률 vs 하이브리드 성능 비교")
print("-" * 80)
print(f"{'Experiment':<25} | {'Acc':<8} | {'F1':<8} | {'Time':<8} | {'Count'}")
print("-" * 80)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    
    pipeline = Pipeline([
        ('vect', TfidfVectorizer(tokenizer=tokenizer, stop_words=stop_list, max_features=5000)),
        ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
    ])
    
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='weighted')
    elapsed = time.time() - start_time
    
    print(f"{name:<25} | {acc:.4f}   | {f1:.4f}   | {elapsed:.1f}s   | {len(stop_set)}")
    
    results.append({
        'Experiment': name,
        'Count': len(stop_set),
        'Accuracy': acc,
        'F1_Score': f1,
        'Time': elapsed
    })

# ---------------------------------------------------------
# 4. 결과 분석 및 저장
# ---------------------------------------------------------
final_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
print("\n[최종 종합 순위]")
print(final_df)

final_df.to_csv("hybrid_benchmark_results.csv", index=False, encoding='utf-8-sig')
print("\n결과 저장 완료: hybrid_benchmark_results.csv")

>>> 일반 불용어(stopwords.txt) 로드 완료: 600개

>>> [확장 실험] 일반 vs 법률 vs 하이브리드 성능 비교
--------------------------------------------------------------------------------
Experiment                | Acc      | F1       | Time     | Count
--------------------------------------------------------------------------------


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1. Baseline (None)        | 0.7101   | 0.7048   | 3.6s   | 0


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


2. Exp A (Zipf)           | 0.6686   | 0.6623   | 2.6s   | 167


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['고등', '상고이', '심', '유서'] not in stop_words.
  warnings.warn(


... (11 outputs left out)

/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '구체', '구토', '근거', '기점', '기준', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후', '힘'] not in stop

9. General + Exp C        | 0.7160   | 0.7119   | 3.4s   | 649


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '구체', '구토', '근거', '기점', '기준', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후', '힘'] not in stop

10. General + Exp E       | 0.7160   | 0.7119   | 3.3s   | 639

[최종 종합 순위]
             Experiment  Count  Accuracy  F1_Score      Time
8    9. General + Exp C    649  0.715976  0.711898  3.361848
9   10. General + Exp E    639  0.715976  0.711898  3.332645
0    1. Baseline (None)      0  0.710059  0.704795  3.621260
3      4. Exp C (MP-VP)     54  0.710059  0.704612  2.596696
4  5. Exp E (Intersect)     44  0.710059  0.704612  2.529668
5       6. General Only    600  0.710059  0.706651  3.646886
7    8. General + Exp B    817  0.698225  0.693893  2.438330
2    3. Exp B (Entropy)    226  0.680473  0.675894  2.557750
6    7. General + Exp A    760  0.674556  0.670076  2.466669
1       2. Exp A (Zipf)    167  0.668639  0.662298  2.603848

결과 저장 완료: hybrid_benchmark_results.csv


In [8]:
# ---------------------------------------------------------
# 최종 산출물 저장: Hybrid (General + Exp E)
# ---------------------------------------------------------

# 1. 리스트 구성 (메모리에 있는 변수 활용)
# set_general: 일반 불용어
# set_e: 법률 교집합 불용어 (44개)
final_hybrid_set = set_general.union(set_e)
final_hybrid_list = sorted(list(final_hybrid_set))

print("="*60)
print(f"🎉 연구 최종 산출물 확정")
print(f"   모델명: Korean Legal Stopwords Hybrid (v1.0)")
print(f"   구성: 일반 불용어({len(set_general)}) + 법률 핵심 불용어({len(set_e)})")
print(f"   총 단어 수: {len(final_hybrid_list)}개")
print("="*60)

# 2. 파일 저장 (.txt)
final_filename = "korean_legal_stopwords_hybrid.txt"

with open(final_filename, "w", encoding="utf-8") as f:
    for word in final_hybrid_list:
        f.write(word + "\n")

print(f"파일 저장 완료: {final_filename}")
print("수고하셨습니다! 이 파일 하나면 한국어 법률 NLP 전처리는 끝입니다.")

🎉 연구 최종 산출물 확정
   모델명: Korean Legal Stopwords Hybrid (v1.0)
   구성: 일반 불용어(600) + 법률 핵심 불용어(44)
   총 단어 수: 639개
파일 저장 완료: korean_legal_stopwords_hybrid.txt
수고하셨습니다! 이 파일 하나면 한국어 법률 NLP 전처리는 끝입니다.
